# RocketPy 6DOF Flight Simulation — Hybrid Rocket

**Vehicle:** SMT033 / L277 Hybrid Motor (N2O + Solid Fuel)
**Flight:** February 13, 2026 — Al Mirfa, Abu Dhabi, UAE
**Actual apogee:** 3,348 m AGL (Fluctus flight computer)

This notebook runs the complete RocketPy 6DOF trajectory simulation using:
- Flight-reconstructed thrust curve (L246, 5520 Ns)
- Real vehicle parameters (measured mass, CG, geometry)
- Real atmospheric data from launch day (Open-Meteo GFS)
- OpenRocket Cd profiles (Mach-dependent)
- Parachute recovery model (6 ft main, CdS = 5.78 m²)

## 1. Installation & Data Files

In [ ]:
!pip install rocketpy -q

In [ ]:
# Download data files from repository
!curl -o Flight_Reconstructed.eng https://raw.githubusercontent.com/jmartos-br/hybrid-rocket-trajectory/main/data/Flight_Reconstructed.eng
!curl -o poweron_drag.csv https://raw.githubusercontent.com/jmartos-br/hybrid-rocket-trajectory/main/data/poweron_drag.csv
!curl -o poweroff_drag.csv https://raw.githubusercontent.com/jmartos-br/hybrid-rocket-trajectory/main/data/poweroff_drag.csv

In [ ]:
from rocketpy import Environment, Rocket, Flight, GenericMotor
import numpy as np
import matplotlib.pyplot as plt

%config InlineBackend.figure_formats = ['svg']
%matplotlib inline

## 2. Environment

Launch site: Al Mirfa, Abu Dhabi, UAE
- Coordinates: 24.18°N, 53.69°E
- Elevation: 5 m ASL
- Atmospheric data: Open-Meteo GFS reanalysis for Feb 13, 2026

In [ ]:
env = Environment(
    latitude=24.18133,
    longitude=53.688379,
    elevation=5,
)
env.set_date((2026, 2, 13, 12))

# Real wind and atmosphere from Open-Meteo GFS
env.set_atmospheric_model(
    type='custom_atmosphere',
    wind_u=[
        (0, 0.07), (10, 0.07), (135, 0.00), (818, -0.45),
        (1542, -0.62), (3164, -0.51), (5854, 12.69),
    ],
    wind_v=[
        (0, -4.00), (10, -4.00), (135, -4.19), (818, -1.46),
        (1542, 1.40), (3164, -0.51), (5854, 4.62),
    ],
    pressure=[
        (0, 101500), (135, 100000), (818, 92500),
        (1542, 85000), (3164, 70000), (5854, 50000),
    ],
    temperature=[
        (0, 302.95), (135, 301.65), (818, 295.25),
        (1542, 288.75), (3164, 281.35), (5854, 264.25),
    ],
)

In [ ]:
env.info()

## 3. Motor

SMT033 hybrid motor — N2O oxidizer + solid fuel grain.

| Parameter | Value |
|-----------|-------|
| Designation | L246 (flight reconstructed) |
| Total impulse | 5,520 Ns |
| Burn time | 22.45 s |
| Peak thrust | 644.8 N |
| Avg thrust | 246 N |
| Dry mass | 6.9 kg |
| Propellant mass | 2.42 kg |
| Chamber | 100 mm dia × 1330 mm |
| Nozzle | 50 mm exit dia |

In [ ]:
motor = GenericMotor(
    thrust_source='./Flight_Reconstructed.eng',
    burn_time=22.455,
    chamber_radius=0.05,
    chamber_height=1.33,
    chamber_position=1.33 / 2,
    propellant_initial_mass=2.42,
    nozzle_radius=0.025,
    dry_mass=6.9,
    dry_inertia=(0.5, 0.5, 0.01),
    nozzle_position=0.0,
    center_of_dry_mass_position=1.33 / 2,
    coordinate_system_orientation='nozzle_to_combustion_chamber',
)

In [ ]:
motor.info()

In [ ]:
motor.thrust.plot(lower=0, upper=23)
plt.title('Flight Reconstructed Thrust Curve — L246')
plt.show()

## 4. Rocket

| Parameter | Value |
|-----------|-------|
| Body diameter | 100 mm |
| Total length | 2.600 m |
| Airframe mass | 3.780 kg |
| Total launch mass | 13.10 kg |
| CG (without motor) | 1.869 m from tail |
| Nose | Ogive, 300 mm |
| Fins | 4× trapezoidal, 0.5° cant |
| Parachute | 6 ft main, CdS = 5.78 m² |

In [ ]:
rocket = Rocket(
    radius=0.05,
    mass=3.780,
    inertia=(3.5, 3.5, 0.005),
    power_off_drag='poweroff_drag.csv',
    power_on_drag='poweron_drag.csv',
    center_of_mass_without_motor=1.869,
    coordinate_system_orientation='tail_to_nose',
)

rocket.add_motor(motor, position=0.0)

rocket.add_nose(length=0.3, kind='ogive', position=2.600)

rocket.add_trapezoidal_fins(
    n=4,
    root_chord=0.145,
    tip_chord=0.065,
    span=0.08,
    sweep_length=0.11,
    cant_angle=0.5,
    position=0.145,
)

rocket.add_tail(
    top_radius=0.05,
    bottom_radius=0.03,
    length=0.055,
    position=0.0,
)

rocket.set_rail_buttons(
    upper_button_position=1.80,
    lower_button_position=0.40,
    angular_position=88,
)

In [ ]:
# Parachute — 6 ft main, Cd = 2.2
# CdS = 2.2 * pi * (1.8288/2)^2 = 5.78 m^2

def main_trigger(p, h, y):
    return True if y[5] < 0 else False

rocket.add_parachute(
    name='Main',
    cd_s=2.2 * 3.14159 * (1.8288 / 2) ** 2,
    trigger=main_trigger,
    sampling_rate=105,
    lag=1.5,
    noise=(0, 8.3, 0.5),
)

In [ ]:
rocket.draw()

In [ ]:
rocket.info()

## 5. Flight Simulation

Launch from 7 m rail at 83.5° inclination, heading 90° (East).

In [ ]:
flight = Flight(
    rocket=rocket,
    environment=env,
    rail_length=7.0,
    inclination=83.5,
    heading=90,
    max_time=600,
    time_overshoot=True,
)

## 6. Results

In [ ]:
apogee_agl = flight.apogee - env.elevation
actual_apogee = 3348  # Fluctus flight computer
error = (apogee_agl - actual_apogee) / actual_apogee * 100

print('=' * 60)
print('SIMULATION RESULTS vs ACTUAL FLIGHT')
print('=' * 60)
print(f'  Simulated apogee  : {apogee_agl:.0f} m AGL')
print(f'  Actual apogee     : {actual_apogee} m AGL (Fluctus)')
print(f'  Error             : {error:+.1f}%')
print(f'  Apogee time       : {flight.apogee_time:.1f} s')
print(f'  Max speed         : {flight.max_speed:.1f} m/s')
print(f'  Max Mach          : {flight.max_mach_number:.2f}')
print(f'  Max acceleration  : {flight.max_acceleration:.1f} m/s²')
print(f'  Impact velocity   : {flight.impact_velocity:.1f} m/s')
print(f'  Flight time       : {flight.t_final:.1f} s')

In [ ]:
flight.info()

## 7. Trajectory Plots

In [ ]:
flight.altitude.plot(lower=0, upper=flight.t_final)
plt.title('Altitude vs Time')
plt.show()

In [ ]:
flight.speed.plot(lower=0, upper=50)
plt.title('Speed vs Time (Ascent)')
plt.show()

In [ ]:
flight.acceleration.plot(lower=0, upper=30)
plt.title('Acceleration During Burn')
plt.show()

In [ ]:
flight.stability_margin.plot(lower=0, upper=30)
plt.title('Stability Margin (calibers)')
plt.show()

In [ ]:
flight.plots.trajectory_3d()

## 8. Export Results

In [ ]:
# Export time series for external comparison
sim_z = np.array(flight.z.source)
sim_spd = np.array(flight.speed.source)
sim_acc = np.array(flight.acceleration.source)

t = sim_z[:, 0]
alt = sim_z[:, 1] - env.elevation
speed = np.interp(t, sim_spd[:, 0], sim_spd[:, 1])
accel = np.interp(t, sim_acc[:, 0], sim_acc[:, 1])

with open('simulation_results.csv', 'w') as f:
    f.write('time_s,altitude_m,speed_ms,acceleration_ms2\n')
    for i in range(len(t)):
        f.write(f'{t[i]:.4f},{alt[i]:.2f},{speed[i]:.2f},{accel[i]:.2f}\n')

print(f'Exported: simulation_results.csv ({len(t)} rows)')

# Download in Colab
try:
    from google.colab import files
    files.download('simulation_results.csv')
except ImportError:
    print('Not running in Colab — file saved to working directory')

## 9. Complete Flight Report

In [ ]:
flight.all_info()